# Deva Voice Agent - Hindi, Hinglish & English

A self-contained notebook voice assistant using **LangGraph**, **LiteLLM**, Gemini with Groq fallbacks, conversation memory, safe sample tools, browser speech recognition/synthesis, and voice-activity barge-in. No separate application frontend or backend is needed. Run cells in order. Browser microphone APIs require a trusted page served from `localhost` or HTTPS. Headphones are recommended to reduce speaker echo during interruption detection.

## 1. Install notebook dependencies
Run once in this notebook environment. Restart the kernel if Jupyter requests it.

In [11]:
# %pip install -q langgraph langchain-core litellm ipywidgets


## 2. Configure the LiteLLM gateway
Keys are requested securely and kept only in this Python kernel's environment. Do not print, paste into notebook source, or save keys in notebook outputs.

In [12]:
import getpass
import os
import logging
logging.getLogger("LiteLLM").setLevel(logging.WARNING)
gemini_key = os.getenv("GEMINI_API_KEY") or os.getenv("GOOGLE_API_KEY")
if not gemini_key:
    gemini_key = getpass.getpass("Gemini API key (input hidden): ").strip()
if not gemini_key:
    raise ValueError("A Gemini API key is required for the primary model.")
os.environ.setdefault("GEMINI_API_KEY", gemini_key)
groq_key = os.getenv("GROQ_API_KEY")
if not groq_key:
    groq_key = getpass.getpass("Groq API key (input hidden; optional fallback): ").strip()
if groq_key:
    os.environ["GROQ_API_KEY"] = groq_key
PRIMARY_MODEL = "gemini/gemini-2.5-flash"
FALLBACK_MODELS = ["groq/llama-3.3-70b-versatile", "groq/openai/gpt-oss-20b"] if groq_key else []
MODEL_CHAIN = [PRIMARY_MODEL, *FALLBACK_MODELS]
print("LiteLLM gateway ready: Gemini primary; Groq fallback " + ("enabled." if FALLBACK_MODELS else "not configured."))


LiteLLM gateway ready: Gemini primary; Groq fallback enabled.


## 3. Define safe sample tools and the LangGraph agent
The calculator accepts arithmetic expressions only. Conversation memory is held in the notebook process and separated by `thread_id`.

In [13]:
import ast
import json
import operator
from datetime import datetime
from zoneinfo import ZoneInfo
import litellm
from langchain_core.messages import AIMessage, HumanMessage, SystemMessage, ToolMessage
from langchain_core.tools import tool
from langchain_core.utils.function_calling import convert_to_openai_tool
from langgraph.checkpoint.memory import MemorySaver
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode
litellm.suppress_debug_info = True
litellm.set_verbose = False
_ALLOWED_BINARY = {ast.Add:operator.add, ast.Sub:operator.sub, ast.Mult:operator.mul, ast.Div:operator.truediv, ast.FloorDiv:operator.floordiv, ast.Mod:operator.mod, ast.Pow:operator.pow}
_ALLOWED_UNARY = {ast.UAdd:operator.pos, ast.USub:operator.neg}
def _safe_arithmetic(node):
    if isinstance(node, ast.Constant) and type(node.value) in (int, float): return node.value
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BINARY:
        right = _safe_arithmetic(node.right)
        if isinstance(node.op, ast.Pow) and abs(right) > 8: raise ValueError("Exponent is limited to 8.")
        value = _ALLOWED_BINARY[type(node.op)](_safe_arithmetic(node.left), right)
        if abs(value) > 10**15: raise ValueError("Result is too large.")
        return value
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_UNARY: return _ALLOWED_UNARY[type(node.op)](_safe_arithmetic(node.operand))
    raise ValueError("Only basic arithmetic with numbers and parentheses is allowed.")
@tool
def calculate(expression: str) -> str:
    """Safely calculate basic arithmetic, for example (12 + 8) * 3."""
    return str(_safe_arithmetic(ast.parse(expression, mode="eval").body))
@tool
def local_time(timezone_name: str = "Asia/Kolkata") -> str:
    """Get the current local time. Defaults to India Standard Time."""
    try: current = datetime.now(ZoneInfo(timezone_name))
    except Exception as exc: raise ValueError(f"Unknown timezone: {timezone_name}") from exc
    return current.strftime("%A, %d %B %Y, %I:%M %p %Z")
@tool
def assistant_capabilities() -> str:
    """Explain what this notebook voice assistant can do."""
    return "I can converse in Hindi, Hinglish, or English; do basic arithmetic; tell local time; and remember this conversation within its thread."
TOOLS = [calculate, local_time, assistant_capabilities]
TOOL_NODE = ToolNode(TOOLS)
OPENAI_TOOLS = [convert_to_openai_tool(item) for item in TOOLS]
SYSTEM_PROMPT = """You are Deva, a friendly, concise multilingual voice assistant. Respond naturally in the language used by the user: Hindi, Hinglish, or English. Understand Devanagari and Hindi written in Latin transliteration. Do not force translation or switch language unnecessarily. Use registered tools when useful. Speak in short, natural sentences suitable for text-to-speech; avoid Markdown tables, emoji, and long preambles. If you do not know, say so clearly."""
def _to_litellm_messages(history):
    result = [{"role":"system", "content":SYSTEM_PROMPT}]
    for item in history:
        if isinstance(item, HumanMessage): result.append({"role":"user", "content":item.content})
        elif isinstance(item, AIMessage):
            message = {"role":"assistant", "content":item.content or None}
            if item.tool_calls:
                message["tool_calls"] = [{"id":call["id"], "type":"function", "function":{"name":call["name"], "arguments":json.dumps(call["args"], ensure_ascii=False)}} for call in item.tool_calls]
            result.append(message)
        elif isinstance(item, ToolMessage): result.append({"role":"tool", "tool_call_id":item.tool_call_id, "content":str(item.content)})
    return result
def agent_node(state):
    messages = _to_litellm_messages(state["messages"])
    errors = []
    for model_name in MODEL_CHAIN:
        try:
            response = litellm.completion(model=model_name, messages=messages, tools=OPENAI_TOOLS, tool_choice="auto", temperature=0.4, timeout=25, num_retries=0)
            message = response.choices[0].message
            calls = []
            for call in (getattr(message, "tool_calls", None) or []):
                raw_args = call.function.arguments or "{}"
                calls.append({"name":call.function.name, "args":json.loads(raw_args) if isinstance(raw_args, str) else raw_args, "id":call.id, "type":"tool_call"})
            print(f"Model used: {model_name}")
            return {"messages":[AIMessage(content=message.content or "", tool_calls=calls)]}
        except Exception as exc:
            errors.append(f"{model_name}: {type(exc).__name__}")
            print(f"{model_name} unavailable; trying next provider.")
    raise RuntimeError("All configured LLM providers failed: " + "; ".join(errors))
def route_after_agent(state):
    last = state["messages"][-1]
    return "tools" if isinstance(last, AIMessage) and last.tool_calls else END
builder = StateGraph(MessagesState)
builder.add_node("agent", agent_node)
builder.add_node("tools", TOOL_NODE)
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", route_after_agent, {"tools":"tools", END:END})
builder.add_edge("tools", "agent")
MEMORY = MemorySaver()
voice_agent = builder.compile(checkpointer=MEMORY)
print("LangGraph ready: START -> Agent <-> Tools -> END (thread memory enabled).")


LangGraph ready: START -> Agent <-> Tools -> END (thread memory enabled).


## 4. Try a text conversation
Reuse a `thread_id` for follow-up context; another ID creates a separate conversation.

In [14]:
def ask_deva(text, thread_id="voice-demo"):
    """Send one turn to the graph; same thread_id preserves its in-memory context."""
    result = voice_agent.invoke(
        {"messages": [HumanMessage(content=text)]},
        config={"configurable": {"thread_id": thread_id}},
    )
    return str(result["messages"][-1].content)

# Opt in to a live provider request by running, for example:
# print(ask_deva("Namaste Deva! 125 ka 4 se guna kitna hota hai?"))
# print(ask_deva("Aur usme 20 jod do."))
print("Conversation helper ready. Uncomment a sample ask_deva(...) call to contact the configured LLM.")


Conversation helper ready. Uncomment a sample ask_deva(...) call to contact the configured LLM.


## 5. Interactive microphone and voice console
Run this cell after the setup cells, then click **Start microphone** and allow the browser permission prompt. Recognized speech is sent automatically after a short pause; you do not need to click **Send to Deva**. If you interrupt Deva while it is speaking, the next recognized phrase is sent together with the prior request, with a brief spoken acknowledgement. **Stop microphone** releases the stream.

In [15]:
import html
import json
import uuid
import ipywidgets as widgets
from IPython.display import Javascript, display
voice_thread_id = "live-voice-session"
voice_history = widgets.HTML(value="<div style='color:#64748b'>Your conversation will appear here.</div>", layout=widgets.Layout(width="100%"))
voice_input = widgets.Textarea(placeholder="Type here, or speak after starting the microphone...", layout=widgets.Layout(width="100%", height="76px"))
voice_input.add_class("voice-input")
voice_output = widgets.Text(value="", layout=widgets.Layout(display="none")); voice_output.add_class("voice-speech-payload")
voice_submit = widgets.Textarea(value="", layout=widgets.Layout(display="none")); voice_submit.add_class("voice-submit-payload")
voice_status = widgets.HTML(value="<b>Status:</b> Ready"); voice_status.add_class("voice-status")
language = widgets.Dropdown(options=[("Auto / Hinglish","auto"),("Hindi","hi-IN"),("Hinglish","en-IN"),("English","en-IN")], value="auto", description="Language:", layout=widgets.Layout(width="230px")); language.add_class("voice-language")
rate = widgets.FloatSlider(value=1.0,min=0.7,max=1.3,step=0.05,description="Speed:",readout_format=".2f",layout=widgets.Layout(width="230px")); rate.add_class("voice-rate")
volume = widgets.FloatSlider(value=1.0,min=0.1,max=1.0,step=0.05,description="Volume:",readout_format=".2f",layout=widgets.Layout(width="230px")); volume.add_class("voice-volume")
barge_in = widgets.Checkbox(value=True,description="Interrupt speech when I start talking"); barge_in.add_class("voice-barge-in")
start_mic = widgets.Button(description="Start microphone",icon="microphone",button_style="success"); start_mic.add_class("voice-start")
stop_mic = widgets.Button(description="Stop microphone",icon="stop",button_style="warning"); stop_mic.add_class("voice-stop")
send_button = widgets.Button(description="Send to Deva",icon="paper-plane",button_style="primary"); send_button.add_class("voice-send")
_voice_turns = []
_busy = False
def _render_history():
    cards=[]
    for role, text in _voice_turns[-10:]:
        safe=html.escape(text).replace("\n","<br>"); color="#e0f2fe" if role=="You" else "#ecfdf5"
        cards.append(f"<div style='background:{color};padding:10px 12px;border-radius:12px;margin:7px 0'><b>{role}</b><br>{safe}</div>")
    voice_history.value="<div style='max-height:300px;overflow:auto;padding:4px'>"+"".join(cards)+"</div>" if cards else "<div style='color:#64748b'>Your conversation will appear here.</div>"
def _send_voice_turn(_=None, text_override=None):
    global _busy
    text=(text_override if text_override is not None else voice_input.value).strip()
    if not text or _busy: return
    _busy=True; voice_input.value=""; _voice_turns.append(("You",text)); _render_history()
    voice_status.value="<b>Status:</b> Thinking..."; send_button.disabled=True
    try:
        answer=ask_deva(text,thread_id=voice_thread_id); _voice_turns.append(("Deva",answer)); _render_history()
        lang=language.value
        if lang=="auto": lang="hi-IN" if any("\u0900"<=ch<="\u097f" for ch in answer) else "en-IN"
        voice_output.value=json.dumps({"text":answer,"lang":lang,"rate":rate.value,"volume":volume.value,"turn":str(uuid.uuid4())})
        voice_status.value="<b>Status:</b> Speaking (interrupt me any time)."
    except Exception as exc: voice_status.value="<b>Status:</b> Request failed: "+html.escape(str(exc))
    finally: _busy=False; send_button.disabled=False
send_button.on_click(_send_voice_turn)
def _on_voice_submit(change):
    text=change["new"].strip()
    if text:
        voice_submit.value=""
        _send_voice_turn(text_override=text)
voice_submit.observe(_on_voice_submit, names="value")
header=widgets.HTML(value="<div style='font-family:system-ui;background:linear-gradient(110deg,#10233e,#087e83);color:white;padding:18px 20px;border-radius:16px;margin-bottom:10px'><div style='font-size:20px;font-weight:750'>Deva Voice Agent</div><div style='opacity:.88;margin-top:4px'>Hindi | Hinglish | English | Gemini + Groq | Barge-in enabled</div></div>")
ui=widgets.VBox([header,voice_status,widgets.HBox([start_mic,stop_mic,send_button]),widgets.HBox([language,rate,volume]),barge_in,voice_input,voice_history,voice_output,voice_submit])
display(ui)
display(Javascript(r"""
(() => {
  if (window.__devaVoice?.cleanup) window.__devaVoice.cleanup();
  const find=s=>document.querySelector(s), status=()=>find('.voice-status');
  const setStatus=t=>{const el=status();if(!el)return;const body=el.querySelector('.widget-html-content')||el;body.textContent='';const label=document.createElement('strong');label.textContent='Status: ';body.append(label,document.createTextNode(t));};
  const input=()=>find('.voice-input textarea');
  const setText=(el,value)=>{Object.getOwnPropertyDescriptor(HTMLTextAreaElement.prototype,'value').set.call(el,value);el.dispatchEvent(new Event('input',{bubbles:true}));el.dispatchEvent(new Event('change',{bubbles:true}));};
  const Recognition=window.SpeechRecognition||window.webkitSpeechRecognition;
  let recognition=null,active=false,speaking=false,interrupted=false,stream=null,context=null,analyser=null,raf=0,retryTimer=0,submitTimer=0,loud=0,lastPayload='',speechBuffer='',lastUserQuery='';
  const language=()=>find('.voice-language select')?.value||'auto';
  const submitSpeech=()=>{const phrase=speechBuffer.trim();if(!phrase)return;speechBuffer='';const wasInterrupted=interrupted;interrupted=false;let message=phrase;if(wasInterrupted&&lastUserQuery)message='Earlier request and follow-up: '+lastUserQuery+'\n\nThe user interrupted the response to add: '+phrase+'\n\nPlease address the entire request, considering both points.';lastUserQuery=wasInterrupted&&lastUserQuery?lastUserQuery+'\nAdditional point: '+phrase:phrase;const payload=find('.voice-submit-payload textarea');if(!payload){setStatus('Could not send recognized speech to the notebook. Rerun the Voice Console cell.');return;}setText(payload,message);setStatus('Heard you. Sending to Deva...');if(wasInterrupted){const chosen=language(),hindi=chosen==='hi-IN'||(chosen==='auto'&&/[\u0900-\u097f]/.test(phrase));const ack=new SpeechSynthesisUtterance(hindi?'Hmm, haan, samajh raha hoon. Aapki pehli aur nayi baat ko saath mein dekhta hoon.':'Hmm, yes, I understand. I will consider both points together.');ack.lang=hindi?'hi-IN':'en-IN';speechSynthesis.cancel();speechSynthesis.speak(ack);}};
  const releaseAudio=()=>{if(raf)cancelAnimationFrame(raf);raf=0;if(stream)stream.getTracks().forEach(t=>t.stop());if(context)context.close().catch(()=>{});stream=context=analyser=null;};
  const stop=()=>{active=false;if(retryTimer)clearTimeout(retryTimer);retryTimer=0;if(recognition){try{recognition.onend=null;recognition.stop();}catch(_){}recognition=null;}releaseAudio();window.speechSynthesis?.cancel();speaking=false;setStatus('Microphone stopped.');};
  const listen=()=>{if(!active||!Recognition)return;const current=new Recognition();recognition=current;current.lang=language()==='auto'?'hi-IN':language();current.continuous=true;current.interimResults=true;
    current.onresult=e=>{let interim='',final='';for(let i=e.resultIndex;i<e.results.length;i++){let t=e.results[i][0].transcript;if(e.results[i].isFinal)final+=t;else interim+=t;}if(speaking&&!interrupted)return;const el=input();if(final.trim()){speechBuffer+=(speechBuffer?' ':'')+final.trim();if(el)setText(el,(speechBuffer+' '+interim).trim());if(submitTimer)clearTimeout(submitTimer);submitTimer=setTimeout(submitSpeech,900);}else if(el&&interim)setText(el,(speechBuffer+' '+interim).trim());};
    current.onerror=e=>{if(['not-allowed','service-not-allowed','audio-capture'].includes(e.error)){active=false;releaseAudio();setStatus('Speech input blocked ('+e.error+'). Allow microphone access for this notebook in browser/site settings, then click Start microphone again.');}else if(!['no-speech','aborted'].includes(e.error))setStatus('Speech recognition error: '+e.error);};current.onend=()=>{if(active&&recognition===current)retryTimer=setTimeout(listen,250);};try{current.start();}catch(e){active=false;releaseAudio();setStatus('Could not start speech recognition: '+(e.message||e.name));}};
  const monitor=async()=>{if(!navigator.mediaDevices?.getUserMedia){setStatus('Speech recognition started; microphone level monitoring is unavailable in this browser context, so barge-in is disabled.');return;}try{const audio=await navigator.mediaDevices.getUserMedia({audio:{echoCancellation:true,noiseSuppression:true,autoGainControl:true}});if(!active){audio.getTracks().forEach(t=>t.stop());return;}stream=audio;const AudioContextClass=window.AudioContext||window.webkitAudioContext;if(!AudioContextClass){setStatus('Speech recognition is active; this browser does not support audio monitoring, so barge-in is disabled.');return;}context=new AudioContextClass();if(context.state==='suspended')await context.resume();const source=context.createMediaStreamSource(stream);analyser=context.createAnalyser();analyser.fftSize=1024;source.connect(analyser);const samples=new Float32Array(analyser.fftSize);
      const tick=()=>{if(!active||!analyser)return;analyser.getFloatTimeDomainData(samples);let sum=0;for(const s of samples)sum+=s*s;let rms=Math.sqrt(sum/samples.length);loud=rms>0.045?loud+1:Math.max(0,loud-1);if(speaking&&find('.voice-barge-in input')?.checked&&loud>=3){speechSynthesis.cancel();speaking=false;interrupted=true;loud=0;setStatus('Interrupted - listening to you...');}raf=requestAnimationFrame(tick);};tick();
    }catch(e){if(active)setStatus('Speech recognition may still work, but audio monitoring/barge-in failed: '+(e.message||e.name)+'. Check this page\'s microphone permission.');}};
  const speak=()=>{let el=find('.voice-speech-payload input');if(!el?.value||el.value===lastPayload)return;lastPayload=el.value;let p;try{p=JSON.parse(el.value);}catch(_){return;}if(!p.text||!window.speechSynthesis)return;speechSynthesis.cancel();let u=new SpeechSynthesisUtterance(p.text);u.lang=p.lang||'en-IN';u.rate=Number(p.rate)||1;u.volume=Number(p.volume)||1;let voices=speechSynthesis.getVoices();u.voice=voices.find(v=>v.lang.toLowerCase()===u.lang.toLowerCase())||voices.find(v=>v.lang.toLowerCase().startsWith(u.lang.slice(0,2).toLowerCase()))||null;u.onstart=()=>{speaking=true;setStatus('Speaking - speak to interrupt.');};u.onend=u.onerror=()=>{speaking=false;setStatus(active?'Listening...':'Ready.');};speechSynthesis.speak(u);};
  const start=()=>{if(active)return;if(!window.isSecureContext){setStatus('Microphone access needs a secure page (HTTPS or localhost). Open the notebook in a supported browser, not an untrusted/insecure page.');return;}if(!Recognition){setStatus('Speech recognition is not supported here. Open this notebook in Chrome or Edge; typing still works.');return;}if(!navigator.mediaDevices?.getUserMedia)setStatus('Starting speech recognition; this context does not expose microphone monitoring, so interruption detection may be unavailable.');else setStatus('Starting microphone - allow the browser permission prompt.');active=true;listen();void monitor();};
  const handleClick=e=>{const target=e.target instanceof Element?e.target:null;if(!target)return;if(target.closest('.voice-start'))start();else if(target.closest('.voice-stop'))stop();};
  const handleKeydown=e=>{if(e.key==='Enter'&&(e.ctrlKey||e.metaKey)&&e.target.closest('.voice-input')){e.preventDefault();find('.voice-send button')?.click();}};
  document.addEventListener('click',handleClick);document.addEventListener('keydown',handleKeydown);let poll=setInterval(speak,250);window.__devaVoice={cleanup:()=>{stop();clearInterval(poll);if(submitTimer)clearTimeout(submitTimer);document.removeEventListener('click',handleClick);document.removeEventListener('keydown',handleKeydown);}};setStatus('Ready - click Start microphone and allow microphone access.');
})();
"""))


<IPython.core.display.Javascript object>

## Troubleshooting and privacy

- Microphone access requires `localhost` or HTTPS and browser permission. Click **Start microphone** and allow the browser prompt. If no prompt appears, allow microphone access in the site/browser settings; use Chrome or Edge. If VS Code's notebook webview blocks microphone access, open the notebook in a browser through Jupyter on `localhost` or HTTPS.
- Use headphones or uncheck interruption detection if speaker echo causes false barge-in. The browser's RMS voice activity detector cancels current speech on sustained microphone input.
- Browser voice availability varies by operating system. Select a language and rate; Hindi speech output requires a Hindi-capable installed browser voice.
- Provider failures: verify Gemini/Groq keys in this kernel and check provider quotas. Keys are not embedded in the notebook.
- `MemorySaver` is in-memory, thread-separated, and cleared when the kernel restarts. Recognized text is sent to the selected LLM provider; microphone audio stays in the browser.
- Click **Stop microphone** before closing the notebook to release the audio stream.